In [1]:
import tempfile
import os
from pathlib import Path
import re

# 📍【关键修复 1】：重定向 Catalyst 的临时文件，彻底解决超算节点硬盘爆满的问题！
custom_tmp_path = os.path.join(os.getcwd(), "catalyst_tmp_cache")
os.makedirs(custom_tmp_path, exist_ok=True)
os.environ['TMPDIR'] = custom_tmp_path
tempfile.tempdir = custom_tmp_path
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import time
import math
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
from scipy.sparse import coo_matrix

import jax
# 开启双精度
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import optax
import pennylane as qml
import catalyst
qml.about()


Name: pennylane
Version: 0.43.1
Summary: PennyLane is a cross-platform Python library for quantum computing, quantum machine learning, and quantum chemistry. Train a quantum computer the same way as a neural network.
Home-page: 
Author: 
Author-email: 
License-Expression: Apache-2.0
Location: /home/lzs/.conda/envs/lzsgpu/lib/python3.12/site-packages
Requires: appdirs, autograd, autoray, cachetools, diastatic-malt, networkx, numpy, packaging, pennylane-lightning, requests, rustworkx, scipy, tomlkit, typing_extensions
Required-by: pennylane_catalyst, pennylane_lightning, pennylane_lightning_gpu

Platform info:           Linux-6.11.0-26-generic-x86_64-with-glibc2.39
Python version:          3.12.12
Numpy version:           2.3.4
Scipy version:           1.16.2
JAX version:             0.6.2
Installed devices:
- default.clifford (pennylane-0.43.1)
- default.gaussian (pennylane-0.43.1)
- default.mixed (pennylane-0.43.1)
- default.qubit (pennylane-0.43.1)
- default.qutrit (pennylane-0.43.1)


In [2]:
# =================== 0. 环境与硬件检查 ===================
print('✅ JAX version:', jax.__version__)
print('✅ Devices:', jax.devices())
if any(d.platform == 'gpu' for d in jax.devices()):
    print('🎉 GPU is working!')
else:
    print('⚠️ No GPU detected')


✅ JAX version: 0.6.2
✅ Devices: [CudaDevice(id=0)]
🎉 GPU is working!


In [3]:
# =================== 1. 数据准备 (直接使用 A100 的大显存优势) ===================
target_l, target_n = 5, 4
matrix_name_candidates = [
    f"L={target_l} N={target_n}.npz",
    f"L={target_l} n={target_n}.npz",
    f"l={target_l} N={target_n}.npz",
    f"l={target_l} n={target_n}.npz",
]
matrix_name = matrix_name_candidates[0]
cwd = Path.cwd()

def _normalize_filename(name: str) -> str:
    return "".join(name.lower().split())

matrix_file = None
matrix_override = os.environ.get("MATRIX_FILE")
if matrix_override:
    override_path = Path(matrix_override).expanduser()
    if override_path.exists():
        matrix_file = override_path
    else:
        print(f"⚠️ MATRIX_FILE 不存在: {override_path}")

candidate_roots = [cwd, cwd / "lunwen", cwd.parent, cwd.parent / "lunwen"]
matrix_candidates = [root / name for root in candidate_roots for name in matrix_name_candidates]
if matrix_file is None:
    matrix_file = next((p for p in matrix_candidates if p.exists()), None)

search_roots = [cwd, cwd.parent, cwd.parent.parent]
if matrix_file is None:
    for root in search_roots:
        if not root.exists():
            continue
        for name in matrix_name_candidates:
            matches = sorted(root.rglob(name))
            if matches:
                matrix_file = matches[0]
                break
        if matrix_file is not None:
            break

if matrix_file is None:
    expected_norms = {_normalize_filename(name) for name in matrix_name_candidates}
    fuzzy_hits = []
    npz_inventory = []
    for root in search_roots:
        if not root.exists():
            continue
        for p in root.rglob("*.npz"):
            npz_inventory.append(p)
            name_norm = _normalize_filename(p.name)
            if name_norm in expected_norms:
                fuzzy_hits.append(p)
                continue
            has_l = re.search(rf"l\\s*=\\s*{target_l}\\b", p.name, flags=re.IGNORECASE)
            has_n = re.search(rf"n\\s*=\\s*{target_n}\\b", p.name, flags=re.IGNORECASE)
            if has_l and has_n:
                fuzzy_hits.append(p)

    if fuzzy_hits:
        matrix_file = sorted(set(fuzzy_hits), key=lambda x: (len(str(x)), str(x)))[0]
    else:
        tried = [str(p) for p in matrix_candidates]
        sample = [str(p) for p in sorted(set(npz_inventory), key=lambda x: str(x))[:20]]
        scanned = [str(p) for p in search_roots]
        raise FileNotFoundError(
            f"Cannot find matrix file for L=5,N=4. cwd={cwd}, tried={tried}, scanned={scanned}, npz_sample={sample}"
        )
H_3 = sp.load_npz(str(matrix_file))
print(f"使用矩阵文件: {matrix_file}")

min_eigvals, _ = eigsh(H_3, k=1, which='SA')
exact_min_energy = float(min_eigvals[0])
print(f"最小特征值 (理论极限): {exact_min_energy:.8f}")

d = H_3.shape[0]
n_qubits = int(np.ceil(np.log2(d)))
l = 2 ** n_qubits
depth = math.ceil(2**n_qubits / n_qubits) + n_qubits
print(f'电路深度: {depth}, 参数总量: {depth * n_qubits}')

# 补全到 2^Nq 并 Gray 编码 (逻辑和之前一样)
H_3_coo = H_3.tocoo()
rows, cols, data = H_3_coo.row.astype(np.int64), H_3_coo.col.astype(np.int64), H_3_coo.data
if l > d:
    rows = np.concatenate([rows, np.arange(d, l, dtype=np.int64)])
    cols = np.concatenate([cols, np.arange(d, l, dtype=np.int64)])
    data = np.concatenate([data, np.full(l - d, 1000.0, dtype=data.dtype)])

def gray_code(n):
    if n == 1: return ["0", "1"]
    lower = gray_code(n - 1)
    return ["0" + x for x in lower] + ["1" + x for x in reversed(lower)]

gray_basis = gray_code(n_qubits)
gray2natural = np.array([int(g, 2) for g in gray_basis], dtype=np.int64)
new_rows, new_cols = gray2natural[rows], gray2natural[cols]
H_gray_csr = coo_matrix((data, (new_rows, new_cols)), shape=(l, l)).tocsr()

# 📍【关键修复 2】：A100 显存管够！直接将 4GB 的矩阵转化为 JAX 密集数组传入 GPU，速度拉满！
H_dense = H_gray_csr.toarray()
H_jax = jnp.array(H_dense, dtype=jnp.complex128)
del H_3, H_3_coo, H_gray_csr, H_dense


使用矩阵文件: /mnt/data/lzs/project/lunwen/L=5 n=4.npz
最小特征值 (理论极限): -35.68407846
电路深度: 354, 参数总量: 4248


In [4]:
# =================== 2. 量子电路与 Catalyst 编译 ===================
hf = np.zeros(n_qubits, dtype=int)
dev = qml.device("lightning.gpu", wires=n_qubits)

# 📍【关键修改 1】：在 qnode 外部提前定义好一个纯 Python 列表！
all_wires = list(range(n_qubits))

# 📍【关键 2】：恢复 H_mat 参数
@qml.qnode(dev)
def cost_circuit(params_2d, H_mat):
    qml.BasisState(hf, wires=all_wires)

    for i in range(n_qubits):
        qml.Hadamard(wires=i)

    for d_idx in range(depth):
        for i in range(n_qubits):
            qml.RY(params_2d[d_idx, i], wires=i)

        for i in range(0, n_qubits - 1, 2):
            qml.CNOT(wires=[i, i + 1])

        qml.CNOT(wires=[n_qubits - 1, 0])

        for i in range(1, n_qubits - 1, 2):
            qml.CNOT(wires=[i, i + 1])

    return qml.expval(qml.Hermitian(H_mat, wires=all_wires))

# =================== 五阶段优化器配置（高精度模式） ===================
seed = 42
key = jax.random.PRNGKey(seed)
# 小随机初始化，避免全零初始化早期陷入对称路径
init_params = 0.005 * jax.random.normal(key, shape=(depth, n_qubits), dtype=jnp.float64)

# Stage-1: 快速下降
schedule_stage1 = optax.exponential_decay(init_value=8e-3, transition_steps=250, decay_rate=0.72)
opt_stage1 = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(learning_rate=schedule_stage1, eps=1e-8),
)

# Stage-2: 主力收敛（降低震荡）
schedule_stage2 = optax.exponential_decay(init_value=1.2e-3, transition_steps=700, decay_rate=0.90)
opt_stage2 = optax.chain(
    optax.clip_by_global_norm(0.12),
    optax.adam(learning_rate=schedule_stage2, eps=1e-8),
)

# Stage-3: 细化收敛
schedule_stage3 = optax.exponential_decay(init_value=6e-5, transition_steps=1200, decay_rate=0.95)
opt_stage3 = optax.chain(
    optax.clip_by_global_norm(0.05),
    optax.adam(learning_rate=schedule_stage3, eps=1e-8),
)

# Stage-4: 超精修
schedule_stage4 = optax.exponential_decay(init_value=1.5e-5, transition_steps=2000, decay_rate=0.97)
opt_stage4 = optax.chain(
    optax.clip_by_global_norm(0.02),
    optax.adam(learning_rate=schedule_stage4, eps=1e-8),
)

# Stage-5: 极限精修（冲击 1e-5 精度）
schedule_stage5 = optax.exponential_decay(init_value=3e-6, transition_steps=3000, decay_rate=0.985)
opt_stage5 = optax.chain(
    optax.clip_by_global_norm(0.008),
    optax.adam(learning_rate=schedule_stage5, eps=1e-8),
)

opt_state_stage1 = opt_stage1.init(init_params)

@qml.qjit(autograph=True)
def update_step_stage1(current_params, current_opt_state, H_matrix):
    def cost_fn(p):
        return cost_circuit(p, H_matrix)

    energy, grads = catalyst.value_and_grad(cost_fn)(current_params)
    updates, next_opt_state = opt_stage1.update(grads, current_opt_state, current_params)
    next_params = optax.apply_updates(current_params, updates)
    return next_params, next_opt_state, energy

@qml.qjit(autograph=True)
def update_step_stage2(current_params, current_opt_state, H_matrix):
    def cost_fn(p):
        return cost_circuit(p, H_matrix)

    energy, grads = catalyst.value_and_grad(cost_fn)(current_params)
    updates, next_opt_state = opt_stage2.update(grads, current_opt_state, current_params)
    next_params = optax.apply_updates(current_params, updates)
    return next_params, next_opt_state, energy

@qml.qjit(autograph=True)
def update_step_stage3(current_params, current_opt_state, H_matrix):
    def cost_fn(p):
        return cost_circuit(p, H_matrix)

    energy, grads = catalyst.value_and_grad(cost_fn)(current_params)
    updates, next_opt_state = opt_stage3.update(grads, current_opt_state, current_params)
    next_params = optax.apply_updates(current_params, updates)
    return next_params, next_opt_state, energy

@qml.qjit(autograph=True)
def update_step_stage4(current_params, current_opt_state, H_matrix):
    def cost_fn(p):
        return cost_circuit(p, H_matrix)

    energy, grads = catalyst.value_and_grad(cost_fn)(current_params)
    updates, next_opt_state = opt_stage4.update(grads, current_opt_state, current_params)
    next_params = optax.apply_updates(current_params, updates)
    return next_params, next_opt_state, energy

@qml.qjit(autograph=True)
def update_step_stage5(current_params, current_opt_state, H_matrix):
    def cost_fn(p):
        return cost_circuit(p, H_matrix)

    energy, grads = catalyst.value_and_grad(cost_fn)(current_params)
    updates, next_opt_state = opt_stage5.update(grads, current_opt_state, current_params)
    next_params = optax.apply_updates(current_params, updates)
    return next_params, next_opt_state, energy

@qml.qjit(autograph=True)
def lbfgs_value_grad(flat_params, H_matrix):
    # 将一维参数还原为 [depth, n_qubits]
    params_2d = jnp.reshape(flat_params, (depth, n_qubits))

    def cost_fn(p):
        return cost_circuit(p, H_matrix)

    energy, grads = catalyst.value_and_grad(cost_fn)(params_2d)
    grads_flat = jnp.reshape(grads, (depth * n_qubits,))
    return energy, grads_flat


In [5]:
# =================== 3. 高精度训练循环（目标 1e-5） ===================
from scipy.optimize import minimize

params = init_params
target_energy = exact_min_energy
target_gap = 1e-5
tolerance = 1e-7
steps = 1000
switch_step_1 = 400   # Stage-1 -> Stage-2
switch_step_2 = 10**9   # 禁用 Stage-2 -> Stage-3（本配置仅跑 Adam 到 1000）
switch_step_3 = 10**9   # 禁用 Stage-3 -> Stage-4
switch_step_4 = 10**9   # 禁用 Stage-4 -> Stage-5

# 若长时间没有刷新 best，就提前切阶段
stagnation_patience_stage1 = 10**9
stagnation_patience_stage2 = 10**9
stagnation_patience_stage3 = 10**9
stagnation_patience_stage4 = 10**9
improve_tol = 1e-12

best_energy = float("inf")
best_gap = float("inf")
best_step = -1
best_params = params

# 保存每一步的能量
energy_history = []

opt_state = opt_state_stage1
current_stage = 1
no_improve_steps = 0

# 分阶段计步（用于打印当前学习率）
stage_step_1 = 0
stage_step_2 = 0
stage_step_3 = 0
stage_step_4 = 0
stage_step_5 = 0

# 梯度裁剪阈值（用于日志显示）
clip_stage_1 = 1.00
clip_stage_2 = 0.12
clip_stage_3 = 0.05
clip_stage_4 = 0.02
clip_stage_5 = 0.008

print("")
print("🚀 开始 Catalyst Adam(1000步) + L-BFGS-B 训练（高精度模式）...")
print(
    "配置: "
    f"S1(lr0=8e-3, clip={clip_stage_1}, decay=0.72/250) | "
    f"S2(lr0=1.2e-3, clip={clip_stage_2}, decay=0.90/700) | "
    "S3-5=disabled"
)
start_time = time.time()

for i in range(steps):
    # 记录“本步实际使用”的阶段与超参数，避免切换时日志错位
    used_stage = current_stage

    if used_stage == 1:
        stage_local_step = stage_step_1
        lr_now = float(schedule_stage1(stage_local_step))
        clip_now = clip_stage_1
        params, opt_state, energy = update_step_stage1(params, opt_state, H_jax)
        stage_step_1 += 1
    elif used_stage == 2:
        stage_local_step = stage_step_2
        lr_now = float(schedule_stage2(stage_local_step))
        clip_now = clip_stage_2
        params, opt_state, energy = update_step_stage2(params, opt_state, H_jax)
        stage_step_2 += 1
    elif used_stage == 3:
        stage_local_step = stage_step_3
        lr_now = float(schedule_stage3(stage_local_step))
        clip_now = clip_stage_3
        params, opt_state, energy = update_step_stage3(params, opt_state, H_jax)
        stage_step_3 += 1
    elif used_stage == 4:
        stage_local_step = stage_step_4
        lr_now = float(schedule_stage4(stage_local_step))
        clip_now = clip_stage_4
        params, opt_state, energy = update_step_stage4(params, opt_state, H_jax)
        stage_step_4 += 1
    else:
        stage_local_step = stage_step_5
        lr_now = float(schedule_stage5(stage_local_step))
        clip_now = clip_stage_5
        params, opt_state, energy = update_step_stage5(params, opt_state, H_jax)
        stage_step_5 += 1

    current_energy = float(energy)
    energy_history.append(current_energy)

    if current_energy < (best_energy - improve_tol):
        best_energy = current_energy
        best_gap = abs(best_energy - exact_min_energy)
        best_step = i
        best_params = params
        no_improve_steps = 0
    else:
        no_improve_steps += 1

    # 阶段切换策略：固定步数 or 停滞触发，并回滚到 best_params
    if current_stage == 1 and (i >= switch_step_1 or no_improve_steps >= stagnation_patience_stage1):
        reason = f"fixed_step={i}" if i >= switch_step_1 else f"plateau={no_improve_steps}"
        params = best_params
        current_stage = 2
        opt_state = opt_stage2.init(params)
        no_improve_steps = 0
        print(f"🔁 切换到 Stage-2, step={i}, reason={reason}, next_lr0=1.2e-3, next_clip={clip_stage_2}")

    elif current_stage == 2 and (i >= switch_step_2 or no_improve_steps >= stagnation_patience_stage2):
        reason = f"fixed_step={i}" if i >= switch_step_2 else f"plateau={no_improve_steps}"
        params = best_params
        current_stage = 3
        opt_state = opt_stage3.init(params)
        no_improve_steps = 0
        print(f"🔁 切换到 Stage-3, step={i}, reason={reason}, next_lr0=6e-5, next_clip={clip_stage_3}")

    elif current_stage == 3 and (i >= switch_step_3 or no_improve_steps >= stagnation_patience_stage3):
        reason = f"fixed_step={i}" if i >= switch_step_3 else f"plateau={no_improve_steps}"
        params = best_params
        current_stage = 4
        opt_state = opt_stage4.init(params)
        no_improve_steps = 0
        print(f"🔁 切换到 Stage-4, step={i}, reason={reason}, next_lr0=1.5e-5, next_clip={clip_stage_4}")

    elif current_stage == 4 and (i >= switch_step_4 or no_improve_steps >= stagnation_patience_stage4):
        reason = f"fixed_step={i}" if i >= switch_step_4 else f"plateau={no_improve_steps}"
        params = best_params
        current_stage = 5
        opt_state = opt_stage5.init(params)
        no_improve_steps = 0
        print(f"🔁 切换到 Stage-5, step={i}, reason={reason}, next_lr0=3e-6, next_clip={clip_stage_5}")

    if i % 100 == 0:
        print(
            f"Step = {i:5d}, Stage = {used_stage}, stage_step = {stage_local_step:5d}, "
            f"lr = {lr_now:.3e}, clip = {clip_now:.3f}, no_imp = {no_improve_steps:4d}, "
            f"Energy = {current_energy:.10f} Ha, Best = {best_energy:.10f} Ha, "
            f"CurrentGap = {(current_energy - exact_min_energy):.6e}, BestGap = {best_gap:.6e}"
        )

    # 达到目标精度自动停止
    if best_gap <= target_gap:
        print("")
        print(f"✅ 达到目标精度: best_gap={best_gap:.3e} <= {target_gap:.1e} (step={i})")
        break

    # 与精确值非常接近也停止
    if abs(current_energy - exact_min_energy) < tolerance:
        print("")
        print(f"🎉 成功收敛！ 在第 {i} 步达到能量: {current_energy:.10f} Ha")
        break

# 训练结束后回滚到历史最优参数
params = best_params





🚀 开始 Catalyst Adam(1000步) + L-BFGS-B 训练（高精度模式）...
配置: S1(lr0=8e-3, clip=1.0, decay=0.72/250) | S2(lr0=1.2e-3, clip=0.12, decay=0.90/700) | S3-5=disabled
Step =     0, Stage = 1, stage_step =     0, lr = 8.000e-03, clip = 1.000, no_imp =    0, Energy = 480.0974332613 Ha, Best = 480.0974332613 Ha, CurrentGap = 5.157815e+02, BestGap = 5.157815e+02
Step =   100, Stage = 1, stage_step =   100, lr = 7.015e-03, clip = 1.000, no_imp =    0, Energy = -17.0755312152 Ha, Best = -17.0755312152 Ha, CurrentGap = 1.860855e+01, BestGap = 1.860855e+01
Step =   200, Stage = 1, stage_step =   200, lr = 6.151e-03, clip = 1.000, no_imp =    0, Energy = -27.7805065773 Ha, Best = -27.7805065773 Ha, CurrentGap = 7.903572e+00, BestGap = 7.903572e+00
Step =   300, Stage = 1, stage_step =   300, lr = 5.394e-03, clip = 1.000, no_imp =    0, Energy = -32.8342882602 Ha, Best = -32.8342882602 Ha, CurrentGap = 2.849790e+00, BestGap = 2.849790e+00
🔁 切换到 Stage-2, step=400, reason=fixed_step=400, next_lr0=1.2e-3, next_

In [6]:
# =================== 4. L-BFGS-B 终局精修（高精度） ===================
lbfgs_enable = best_gap > target_gap
target_gap = 1e-5
# 先做可行性探针：小步数验证 fun/jac 数值稳定，再决定是否长程精修
lbfgs_probe_enable = True
lbfgs_probe_maxiter = 8

lbfgs_maxiter = 6000
lbfgs_maxcor = 50
lbfgs_maxls = 50
lbfgs_ftol = 1e-15
lbfgs_gtol = 1e-12
lbfgs_log_every = 20

lbfgs_energy_history = []
lbfgs_stop_on_target_gap = True
lbfgs_stop_gap = target_gap
lbfgs_stop_state = {"triggered": False, "x": None, "f": None}

if lbfgs_enable:
    print("\n🚀 进入 L-BFGS-B 终局精修...")
    print(
        f"L-BFGS-B 配置: maxiter={lbfgs_maxiter}, ftol={lbfgs_ftol:.1e}, "
        f"gtol={lbfgs_gtol:.1e}, maxcor={lbfgs_maxcor}, maxls={lbfgs_maxls}"
    )

    x0 = np.asarray(best_params, dtype=np.float64).reshape(-1)

    # 缓存最近一次 fun/jac，避免 scipy 在同一点重复求值
    cache = {"x": None, "f": None, "g": None}

    def _fun_jac_cached(x_np):
        x_cached = cache["x"]
        if x_cached is None or x_cached.shape != x_np.shape or not np.array_equal(x_cached, x_np):
            x_jax = jnp.asarray(x_np, dtype=jnp.float64)
            e_val, g_val = lbfgs_value_grad(x_jax, H_jax)
            cache["x"] = np.array(x_np, copy=True)
            cache["f"] = float(e_val)
            cache["g"] = np.asarray(g_val, dtype=np.float64)
        return cache["f"], cache["g"]

    def _scipy_fun(x_np):
        f_val, _ = _fun_jac_cached(x_np)
        return f_val

    def _scipy_jac(x_np):
        _, g_val = _fun_jac_cached(x_np)
        return g_val

    def _scipy_callback(xk):
        f_val, _ = _fun_jac_cached(xk)
        lbfgs_energy_history.append(float(f_val))
        gap_now = abs(f_val - exact_min_energy)
        if len(lbfgs_energy_history) == 1 or len(lbfgs_energy_history) % lbfgs_log_every == 0:
            print(
                f"[L-BFGS] iter={len(lbfgs_energy_history):4d}, "
                f"Energy={f_val:.10f} Ha, Gap={gap_now:.6e}"
            )
        if lbfgs_stop_on_target_gap and gap_now <= lbfgs_stop_gap:
            lbfgs_stop_state["triggered"] = True
            lbfgs_stop_state["x"] = np.array(xk, dtype=np.float64, copy=True)
            lbfgs_stop_state["f"] = float(f_val)
            print(f"[L-BFGS] 提前停止: gap={gap_now:.6e} <= target_gap={lbfgs_stop_gap:.1e}")
            raise StopIteration

    f0, _ = _fun_jac_cached(x0)
    print(f"[L-BFGS] 初始能量: {f0:.10f} Ha, Gap={(f0 - exact_min_energy):.6e}")

    # ---------- 可行性探针 ----------
    if lbfgs_probe_enable:
        probe_res = minimize(
            fun=_scipy_fun,
            x0=x0,
            jac=_scipy_jac,
            method='L-BFGS-B',
            options={
                'maxiter': lbfgs_probe_maxiter,
                'maxcor': 10,
                'maxls': 20,
                'ftol': 1e-9,
                'gtol': 1e-8,
            },
        )

        probe_f = float(probe_res.fun)
        probe_gap = abs(probe_f - exact_min_energy)
        probe_delta = f0 - probe_f

        print(
            f"[L-BFGS Probe] success={probe_res.success}, nit={probe_res.nit}, "
            f"Energy={probe_f:.10f} Ha, Gap={probe_gap:.6e}, ΔE={probe_delta:.3e}"
        )

        # 可行性判据：数值有限，且没有明显恶化
        probe_ok = np.isfinite(probe_f) and (probe_delta > -1e-8)
        if probe_ok:
            x0 = np.asarray(probe_res.x, dtype=np.float64)
        else:
            lbfgs_enable = False
            print("⚠️ L-BFGS 探针未通过，跳过长程 L-BFGS-B 精修。")

    # ---------- 长程精修 ----------
    if lbfgs_enable:
        lbfgs_stopped_early = False
        try:
            res = minimize(
                fun=_scipy_fun,
                x0=x0,
                jac=_scipy_jac,
                method='L-BFGS-B',
                callback=_scipy_callback,
                options={
                    'maxiter': lbfgs_maxiter,
                    'maxcor': lbfgs_maxcor,
                    'maxls': lbfgs_maxls,
                    'ftol': lbfgs_ftol,
                    'gtol': lbfgs_gtol,
                },
            )
        except StopIteration:
            lbfgs_stopped_early = lbfgs_stop_state["triggered"]
            res = None

        if lbfgs_stopped_early:
            lbfgs_final_energy = float(lbfgs_stop_state["f"])
            lbfgs_x = np.asarray(lbfgs_stop_state["x"], dtype=np.float64)
            lbfgs_success = True
            lbfgs_status = 99
            lbfgs_nit = len(lbfgs_energy_history)
            lbfgs_nfev = lbfgs_nit
            lbfgs_njev = lbfgs_nit
        else:
            lbfgs_final_energy = float(res.fun)
            lbfgs_x = np.asarray(res.x, dtype=np.float64)
            lbfgs_success = bool(res.success)
            lbfgs_status = int(res.status)
            lbfgs_nit = int(res.nit)
            lbfgs_nfev = int(res.nfev)
            lbfgs_njev = int(res.njev)

        lbfgs_final_gap = abs(lbfgs_final_energy - exact_min_energy)

        # 保证最终值进入曲线（避免重复追加）
        if (not lbfgs_energy_history) or abs(lbfgs_energy_history[-1] - lbfgs_final_energy) > 1e-15:
            lbfgs_energy_history.append(lbfgs_final_energy)
        energy_history.extend(lbfgs_energy_history)

        if lbfgs_final_energy < best_energy:
            best_energy = lbfgs_final_energy
            best_gap = lbfgs_final_gap
            best_step = steps + len(lbfgs_energy_history) - 1
            best_params = jnp.asarray(lbfgs_x, dtype=jnp.float64).reshape((depth, n_qubits))

        params = best_params

        print(
            f"[L-BFGS] 结束: success={lbfgs_success}, status={lbfgs_status}, "
            f"nit={lbfgs_nit}, nfev={lbfgs_nfev}, njev={lbfgs_njev}"
        )
        print(f"[L-BFGS] 最终能量: {lbfgs_final_energy:.10f} Ha, Gap={lbfgs_final_gap:.6e}")
else:
    print("\nℹ️ best_gap 已达到目标，跳过 L-BFGS-B 精修。")


# 将所有能量转换为数组
energy_array = np.array(energy_history, dtype=np.float64)

print("-" * 30)
print(f"训练总耗时: {time.time() - start_time:.2f} 秒")
print(f"最后一步能量: {current_energy:.10f} Ha")
print(f"历史最优能量: {best_energy:.10f} Ha (step={best_step})")
print(f"最优能量差(best_gap): {best_gap:.8e} Ha")
print(f"目标精度(target_gap): {target_gap:.1e} Ha")
print(f"energy_array 已生成，shape={energy_array.shape}")


🚀 进入 L-BFGS-B 终局精修...
L-BFGS-B 配置: maxiter=6000, ftol=1.0e-15, gtol=1.0e-12, maxcor=50, maxls=50
[L-BFGS] 初始能量: -35.5767688452 Ha, Gap=1.073096e-01
[L-BFGS Probe] success=False, nit=8, Energy=-35.5928439297 Ha, Gap=9.123453e-02, ΔE=1.608e-02
[L-BFGS] iter=   1, Energy=-35.5928838172 Ha, Gap=9.119465e-02
[L-BFGS] iter=  20, Energy=-35.6121515181 Ha, Gap=7.192695e-02
[L-BFGS] iter=  40, Energy=-35.6281135201 Ha, Gap=5.596494e-02
[L-BFGS] iter=  60, Energy=-35.6356183310 Ha, Gap=4.846013e-02
[L-BFGS] iter=  80, Energy=-35.6417490563 Ha, Gap=4.232941e-02
[L-BFGS] iter= 100, Energy=-35.6490490539 Ha, Gap=3.502941e-02
[L-BFGS] iter= 120, Energy=-35.6545743132 Ha, Gap=2.950415e-02
[L-BFGS] iter= 140, Energy=-35.6585032147 Ha, Gap=2.557525e-02
[L-BFGS] iter= 160, Energy=-35.6624671881 Ha, Gap=2.161128e-02
[L-BFGS] iter= 180, Energy=-35.6653390651 Ha, Gap=1.873940e-02
[L-BFGS] iter= 200, Energy=-35.6678723892 Ha, Gap=1.620608e-02
[L-BFGS] iter= 220, Energy=-35.6700494237 Ha, Gap=1.402904e-02
[

In [1]:
# =================== 4. 能量差距对数图（Log） ===================
import numpy as np
import matplotlib.pyplot as plt

if 'energy_array' not in globals():
    raise RuntimeError("未检测到 energy_array，请先运行训练单元。")

energy_plot = np.asarray(energy_array, dtype=np.float64)
if energy_plot.size == 0:
    raise RuntimeError("energy_array 为空，无法绘图。")

# 使用 1..N 作为横轴，便于在需要时切换 log-x
steps_axis = np.arange(1, energy_plot.size + 1)

# 历史最优包络线（单调不增）
best_so_far = np.minimum.accumulate(energy_plot)

# 构造可取对数的 gap（必须 > 0）
eps = 1e-14
if 'exact_min_energy' in globals():
    ref_energy = float(exact_min_energy)
    ref_name = 'exact_min_energy'
    gap = np.maximum(energy_plot - ref_energy, eps)
    best_gap = np.maximum(best_so_far - ref_energy, eps)
else:
    # 若没有精确值，则以最终最优值作参考
    ref_energy = float(best_so_far[-1])
    ref_name = 'best_final'
    gap = np.maximum(energy_plot - ref_energy + eps, eps)
    best_gap = np.maximum(best_so_far - ref_energy + eps, eps)

# 后段放大，观察精修阶段
trail_n = max(200, energy_plot.size // 4)
start_tail = max(0, energy_plot.size - trail_n)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

# 左图：全程 gap 对数图（y-log）
axes[0].semilogy(steps_axis, gap, lw=1.2, alpha=0.8, label='Gap (Energy - Ref)')
axes[0].semilogy(steps_axis, best_gap, lw=1.4, label='Best Gap')
axes[0].set_title(f'Log Gap vs Step (Full), ref={ref_name}')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Gap (Ha, log scale)')
axes[0].grid(alpha=0.25, which='both')
axes[0].legend()

# 右图：后段 gap 对数图（y-log）
axes[1].semilogy(steps_axis[start_tail:], gap[start_tail:], lw=1.2, alpha=0.85, label='Gap (tail)')
axes[1].semilogy(steps_axis[start_tail:], best_gap[start_tail:], lw=1.4, label='Best Gap (tail)')
axes[1].set_title(f'Log Gap vs Step (Last {energy_plot.size - start_tail} Steps)')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Gap (Ha, log scale)')
axes[1].grid(alpha=0.25, which='both')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"参考能量(ref): {ref_energy:.10f} Ha ({ref_name})")
print(f"最终能量: {energy_plot[-1]:.10f} Ha")
print(f"历史最优: {best_so_far[-1]:.10f} Ha")
print(f"最终 gap: {gap[-1]:.10e} Ha")
print(f"最优 gap: {best_gap[-1]:.10e} Ha")


RuntimeError: 未检测到 energy_array，请先运行训练单元。